# Persona Cloner — Standalone LLM-as-a-Judge Evaluator

Evaluates a cloned persona (`*_system_prompt.txt` or `*_character_card.json`) against **held-out WhatsApp turns** using a 4-dimension computational linguistics & forensic stylistics rubric:
1. **Orthographic & Micro-Stylistic Fidelity (`style_score`)** — casing, punctuation, length/burstiness, elongation, laughter, emojis
2. **Sociolinguistic & Register Alignment (`register_score`)** — pronoun register (`gue/lu`, `aku/kamu`, `saya/anda`), slang, code-switching
3. **Personality & Attitude Consistency (`persona_score`)** — humor, warmth, sarcasm, bluntness
4. **Conversational Coherence & Relevance (`coherence_score`)** — logical, contextually appropriate reply

Works standalone in **Google Colab** or **local Jupyter**.

In [ ]:
import getpass
import json
import os
import random
import re
import statistics
import pandas as pd
import requests

print("Libraries ready. pandas", pd.__version__)

## 1. Configuration

In [ ]:
# === Input Files ===
CHAT_PATH = None          # Path to WhatsApp .txt export (None = upload in Colab)
SYSTEM_PROMPT_PATH = None # Path to *_system_prompt.txt or *_character_card.json (None = paste below or upload)
SYSTEM_PROMPT_INLINE = """"""  # Optional: paste system prompt directly if SYSTEM_PROMPT_PATH is None

# === Speaker Settings ===
TARGET_NAME = "Mogu"      # Exact sender name in the WhatsApp export
CHAR_NAME = "Fossetta"    # Character name used in the system prompt
USER_NAME = "You"         # Name used for {{user}}
USER_ALIASES = []         # e.g. ["Pais*", "Bro"]
CHAR_ALIASES = []
THIRD_PARTY_NAMES = []

# === LLM & Judge Settings ===
PROVIDER = "openrouter"   # "openrouter", "openai", "fireworks", or "custom"
BASE_URL = None           # Only needed if PROVIDER == "custom"
CANDIDATE_MODEL = "deepseek/deepseek-v4.1-flash"
JUDGE_MODEL = "openai/gpt-4o"  # Can be the same as CANDIDATE_MODEL or a stronger judge model

# === Evaluation Settings ===
NUM_SAMPLES = 5           # Number of held-out turns to evaluate
TEST_RATIO = 0.2          # Chronological held-out split (last 20% of messages)
SESSION_GAP_HOURS = 6
SEED = 42
OUTPUT_PATH = "evaluation_report.json"

In [ ]:
# Load API key from Colab Secrets, environment variable, or prompt
API_KEY = os.environ.get("OPENROUTER_API_KEY") or os.environ.get("OPENAI_API_KEY") or ""
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPENROUTER_API_KEY") or ""
    except Exception:
        pass
if not API_KEY and PROVIDER != "custom":
    API_KEY = getpass.getpass(f"Enter API key for {PROVIDER}: ").strip()

print("API key configured:", bool(API_KEY))

## 2. Load WhatsApp Chat & System Prompt

In [ ]:
def _upload_colab_file(prompt_msg):
    from google.colab import files
    print(prompt_msg)
    uploaded = files.upload()
    fname = next(iter(uploaded))
    return uploaded[fname].decode("utf-8", errors="ignore")

if CHAT_PATH:
    with open(CHAT_PATH, "r", encoding="utf-8", errors="ignore") as f:
        RAW_CHAT_TEXT = f.read()
else:
    RAW_CHAT_TEXT = _upload_colab_file("Upload your WhatsApp chat .txt export:")

if SYSTEM_PROMPT_INLINE.strip():
    SYSTEM_PROMPT = SYSTEM_PROMPT_INLINE.strip()
elif SYSTEM_PROMPT_PATH:
    with open(SYSTEM_PROMPT_PATH, "r", encoding="utf-8", errors="ignore") as f:
        raw_sp = f.read()
    if SYSTEM_PROMPT_PATH.lower().endswith(".json"):
        card = json.loads(raw_sp)
        data = card.get("data", card)
        SYSTEM_PROMPT = data.get("description", "")
        if data.get("mes_example"):
            SYSTEM_PROMPT += "\n\n## CONTOH SEGMEN BALASAN ASLI\n" + data["mes_example"]
    else:
        SYSTEM_PROMPT = raw_sp.strip()
else:
    raw_sp = _upload_colab_file("Upload your *_system_prompt.txt or *_character_card.json:")
    try:
        card = json.loads(raw_sp)
        data = card.get("data", card)
        SYSTEM_PROMPT = data.get("description", raw_sp).strip()
    except json.JSONDecodeError:
        SYSTEM_PROMPT = raw_sp.strip()

print(f"Loaded chat ({len(RAW_CHAT_TEXT):,} chars) and system prompt ({len(SYSTEM_PROMPT):,} chars).")

## 3. WhatsApp Parser, Anonymizer & Held-Out Split

In [ ]:
DATE = r"(?P<date>\d{1,4}[-/.]\d{1,2}(?:[-/.]\d{1,4})?)"
TIME = r"(?P<time>\d{1,2}[.:]\d{2}(?:[.:]\d{2})?(?:\s?[AaPp]\.?\s?[Mm]\.?)?)"
HEADER_PATTERNS = [
    re.compile(rf"^\[{DATE},?\s+{TIME}\]\s*(?P<rest>.*)$"),
    re.compile(rf"^{DATE},?\s+{TIME}\s*[-\u2013\u2014]\s*(?P<rest>.*)$"),
    re.compile(rf"^\[{TIME}\]\s*(?P<rest>.*)$"),
    re.compile(rf"^{TIME}\s*[-\u2013\u2014]\s*(?P<rest>.*)$"),
]
RE_SENDER = re.compile(r"^([^:\n]{1,60}?):\s(.*)$", re.S)
NOISE = [
    r"media (omitted|tidak disertakan)",
    r"(gambar|video|audio|stiker|sticker|gif|dokumen|image|document|kontak|contact card) (omitted|tidak disertakan|dihilangkan)",
    r"pesan ini dihapus", r"this message was deleted", r"anda menghapus pesan ini",
    r"you deleted this message", r"^null$", r"menunggu pesan ini", r"waiting for this message",
]

def is_noise(t):
    return len(t) < 80 and any(re.search(p, t.lower()) for p in NOISE)

def is_pasted(t, max_len=400, max_words=60):
    t = str(t)
    return len(t.split()) > max_words or len(t) > max_len or t.count("\n") >= 3 or "```" in t or t.count("**") >= 2

def _norm_time(t):
    t = re.sub(r"\s+", "", (t or "").replace("\u202f", ""))
    m = re.match(r"^(\d{1,2})[.:](\d{2})(?:[.:](\d{2}))?([AaPp]\.?[Mm]\.?)?$", t)
    if not m:
        return ""
    h, mi, sec, ap = int(m.group(1)), m.group(2), m.group(3), m.group(4)
    if ap:
        ap = ap.replace(".", "").lower()
        if ap == "pm" and h < 12:
            h += 12
        if ap == "am" and h == 12:
            h = 0
    return f"{h:02d}:{mi}" + (f":{sec}" if sec else "")

def _pisah(raw):
    return [int(x) for x in re.split(r"[-/.]", raw) if x.isdigit()]

def _deteksi_urutan(dates):
    f, s = [], []
    for d in dates:
        if not d:
            continue
        p = _pisah(d)
        if len(p) == 3 and p[0] > 99:
            return False
        if len(p) >= 2:
            f.append(p[0]); s.append(p[1])
    if f and max(f) > 12:
        return True
    if s and max(s) > 12:
        return False
    return True

def _ke_iso(raw, time_val, dayfirst, default_year):
    p = _pisah(raw) if raw else [1, 1]
    if len(p) == 3 and p[0] > 99:
        y, mo, d = p
    elif len(p) == 3:
        a, b, y = p
        y += (2000 if y < 70 else 1900) if y < 100 else 0
        d, mo = (a, b) if dayfirst else (b, a)
    else:
        a, b = (p + [1, 1])[:2]
        d, mo = (a, b) if dayfirst else (b, a)
        y = default_year
    if not (1 <= mo <= 12):
        d, mo = mo, d
    return f"{y:04d}-{mo:02d}-{d:02d}" + (f" {time_val}" if time_val else "")

def parse_whatsapp_text(raw_text):
    for ch in ["\u200e", "\u200f", "\ufeff", "\u202f", "\u00a0"]:
        raw_text = raw_text.replace(ch, " " if ch in ["\u202f", "\u00a0"] else "")
    rows, cur = [], None
    for line in raw_text.splitlines():
        m, rest = None, None
        for pat in HEADER_PATTERNS:
            mm = pat.match(line)
            if mm:
                m, rest = mm, mm.group("rest")
                break
        if m is not None:
            if cur:
                rows.append(cur)
            cur = None
            ms = RE_SENDER.match(rest)
            if ms:
                g = m.groupdict()
                cur = {"date": g.get("date") or "", "time": _norm_time(g.get("time") or ""), "sender": ms.group(1).strip(), "text": ms.group(2).strip()}
        elif cur is not None:
            cur["text"] += "\n" + line.strip()
    if cur:
        rows.append(cur)
    df = pd.DataFrame(rows)
    dayfirst = _deteksi_urutan(df["date"].tolist())
    default_year = pd.Timestamp.today().year
    df["datetime"] = pd.to_datetime([_ke_iso(r["date"], r["time"], dayfirst, default_year) for _, r in df.iterrows()], errors="coerce").fillna(pd.Timestamp.today())
    df = df.sort_values("datetime").reset_index(drop=True)
    df["text"] = df["text"].str.replace(r"\s*<(pesan ini diedit|this message was edited)>", "", regex=True, flags=re.I).str.strip()
    return df[(df["text"] != "") & (~df["text"].apply(is_noise))].reset_index(drop=True)

def anonymize_messages(df, target_name, user_aliases=None, char_aliases=None, third_party_names=None):
    df = df.copy()
    target_sender = next((s for s in df["sender"].unique() if s.strip().lower() == target_name.strip().lower()), target_name)
    others = [s for s in df["sender"].unique() if s != target_sender]
    df["sender"] = df["sender"].apply(lambda s: "TARGET" if s == target_sender else "LAWAN")
    def name_regex(names):
        forms = set()
        for n in names:
            n = (n or "").strip()
            if n.endswith("*") and len(n) >= 4:
                forms.add(re.escape(n[:-1]) + r"\w*")
            elif len(n) >= 3 and re.search(r"[A-Za-z]", n):
                forms.add(re.escape(n) + re.escape(n[-1]) + r"*(?:chan|kun|san|sama|nya|ku|mu)?")
        forms = sorted(forms, key=len, reverse=True)
        return re.compile(r"(?<!\w)(?:" + "|".join(forms) + r")(?!\w)", re.I) if forms else None
    char_rx = name_regex({target_sender} | set(char_aliases or []))
    user_rx = name_regex(set(others) | set(user_aliases or []))
    third_rx = name_regex(set(third_party_names or []))
    re_url = re.compile(r"(?:https?://|www\.)\S+", re.I)
    re_email = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
    re_phone = re.compile(r"(?:\+?62|\b0)[\s-]?8\d{1,3}[\s-]?\d{3,4}[\s-]?\d{2,5}|\b\d{9,}\b")
    re_handle = re.compile(r"(?<![\w.])@[A-Za-z0-9._]{3,30}\b")
    def scrub(t):
        t = re_handle.sub("<AKUN>", re_phone.sub("<NOMOR>", re_email.sub("<EMAIL>", re_url.sub("<URL>", str(t)))))
        if third_rx: t = third_rx.sub("<NAMA>", t)
        if user_rx: t = user_rx.sub("{{user}}", t)
        if char_rx: t = char_rx.sub("{{char}}", t)
        return t
    df["text"] = df["text"].apply(scrub)
    return df

def extract_turn_pairs(df, session_gap_hours=6):
    d = df.copy().reset_index(drop=True)
    d["turn"] = (d.sender != d.sender.shift()).cumsum()
    d["session"] = (d.datetime.diff() > pd.Timedelta(hours=session_gap_hours)).cumsum()
    turns = [(s, sess, list(g.text)) for (_, s, sess), g in d.groupby(["turn", "sender", "session"], sort=False)]
    pairs = []
    for i in range(1, len(turns)):
        ps, psess, ptxt = turns[i - 1]
        s, sess, txt = turns[i]
        if s == "TARGET" and ps != "TARGET" and sess == psess:
            ctx = "\n".join(str(x) for x in ptxt[-2:])[:300]
            rep = "\n".join(str(x) for x in txt[:5] if str(x).strip())
            if 0 < len(rep) <= 400 and not re.search(r"<[A-Z_]+>", ctx + " " + rep) and not any(is_pasted(x) for x in ptxt[-2:] + txt):
                pairs.append({"konteks": ctx, "balasan": rep})
    return pairs

df_raw = parse_whatsapp_text(RAW_CHAT_TEXT)
df_anon = anonymize_messages(df_raw, TARGET_NAME, USER_ALIASES, CHAR_ALIASES, THIRD_PARTY_NAMES)
split_idx = int(len(df_anon) * (1 - TEST_RATIO))
train_df = df_anon.iloc[:split_idx].reset_index(drop=True)
test_df = df_anon.iloc[split_idx:].reset_index(drop=True)
test_pairs = extract_turn_pairs(test_df, SESSION_GAP_HOURS) or extract_turn_pairs(df_anon, SESSION_GAP_HOURS)

# Sample reference messages from train_df
ref_candidates = [str(t).strip() for t in train_df[train_df["sender"] == "TARGET"]["text"] if str(t).strip() and not is_pasted(t) and not re.search(r"<[A-Z_]+>", str(t))]
random.Random(SEED).shuffle(ref_candidates)
reference_samples = list(dict.fromkeys(ref_candidates))[:10]

print(f"Total messages: {len(df_anon)} | Held-out pairs: {len(test_pairs)} | Reference samples: {len(reference_samples)}")

## 4. LLM Client & Forensic Judge Rubric

In [ ]:
SCORE_KEYS = ("style_score", "register_score", "persona_score", "coherence_score")

JUDGE_SYSTEM_PROMPT = (
    "You are an expert computational linguist and forensic stylistician evaluating an AI persona "
    "simulation system on Indonesian/multilingual instant messaging (WhatsApp). "
    "Evaluate objectively based strictly on the provided inputs. "
    "Output ONLY valid JSON matching the requested schema, without markdown fences or extra text."
)

JUDGE_USER_TEMPLATE = """Your task is to evaluate how accurately a **Candidate Response** mimics the **Target Speaker** compared to the **Ground-Truth Human Response** and **Real Reference Samples** from that speaker.

### Evaluation Inputs
1. **Conversation Context (LAWAN's prompt/turn):**
{conversation_context}

2. **Real Reference Samples of TARGET's Natural Writing Style:**
{target_reference_samples}

3. **Ground-Truth Human Response (What TARGET actually replied in this held-out turn):**
{ground_truth_reply}

4. **Candidate Model Response (To be evaluated):**
{candidate_response}

---

### Scoring Rubric (Score each dimension from 1 to 5)

**1. Orthographic & Micro-Stylistic Fidelity (`style_score`)**
Does the candidate match the Target's exact mechanics—casing (e.g., all-lowercase), punctuation absence/quirks, message length/burstiness, expressive lengthening (e.g., "iyaaa"), written laughter (e.g., "wkwk", "awok"), and emojis?
- **1:** Sounds like a standard, polite, properly capitalized AI assistant.
- **2:** Minor attempt at casual tone, but clearly violates the Target's casing, length, or punctuation habits.
- **3:** Moderately casual, captures a few surface habits, but feels generic or inconsistent.
- **4:** Closely matches the Target's formatting, casing, and length distribution with minor deviations.
- **5:** Indistinguishable from the Target's orthographic fingerprint and message rhythm.

**2. Sociolinguistic & Register Alignment (`register_score`)**
Does the candidate use the Target's authentic Indonesian/English code-switching ratio, slang lexicon, and pronoun register (e.g., gue/lu vs. aku/kamu vs. saya/anda)?
- **1:** Wrong pronoun register or overly formal/textbook language that the Target never uses.
- **2:** Frequent register slips or unnatural slang usage.
- **3:** Acceptable register, though vocabulary feels flatter than the Target's real samples.
- **4:** Natural use of the Target's pronouns, slang, and code-switching habits.
- **5:** Native-level mastery of the Target's exact sociolect and lexical choices.

**3. Personality & Attitude Consistency (`persona_score`)**
Comparing against the Ground-Truth Human Response and Reference Samples, does the candidate react with the same underlying attitude, humor, warmth, sarcasm, or bluntness?
- **1:** Completely wrong emotional stance (e.g., overly helpful/empathetic when Target is dry/sarcastic).
- **2:** Generic conversational attitude that misses the Target's distinct personality traits.
- **3:** Plausible reaction, partially aligns with the Target's temperament.
- **4:** Strongly reflects the Target's typical attitude, humor, and conversational stance.
- **5:** Captures the exact interpersonal vibe and psychological reaction of the Target.

**4. Conversational Coherence & Relevance (`coherence_score`)**
Regardless of style, is the reply a logical, contextually appropriate response to LAWAN's message?
- **1:** Nonsensical, off-topic, or hallucinates contradictory context.
- **2:** Tangential or awkward transition.
- **3:** Understandable reply, though slightly generic.
- **4:** Natural, contextually appropriate reply.
- **5:** Seamless, natural conversational turn.

---

### Instructions
First, write a brief, objective analysis comparing the Candidate Response against the Ground-Truth Reply and Reference Samples across orthography, register, and attitude inside the JSON `"analysis"` field. Output ONLY valid JSON matching the schema below:

{
  "analysis": "<2-3 sentences comparing specific linguistic and behavioral markers>",
  "style_score": <int 1-5>,
  "register_score": <int 1-5>,
  "persona_score": <int 1-5>,
  "coherence_score": <int 1-5>
}"""

def call_llm(messages, model, temperature=0.3, max_tokens=1000):
    endpoints = {
        "openrouter": "https://openrouter.ai/api/v1/chat/completions",
        "openai": "https://api.openai.com/v1/chat/completions",
        "fireworks": "https://api.fireworks.ai/inference/v1/chat/completions",
    }
    url = endpoints.get(PROVIDER) or (BASE_URL.rstrip("/") + ("" if BASE_URL.rstrip("/").endswith("/chat/completions") else "/chat/completions"))
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["Authorization"] = f"Bearer {API_KEY}"
    r = requests.post(url, headers=headers, json={"model": model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens}, timeout=120)
    r.raise_for_status()
    return r.json()["choices"][0]["message"].get("content") or ""

def parse_json_loose(text):
    t = re.sub(r"<think>.*?</think>", "", str(text or ""), flags=re.S | re.I).strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", t, flags=re.S | re.I)
    t = fence.group(1).strip() if fence else t
    s, e = t.find("{"), t.rfind("}")
    s_str = re.sub(r",\s*([}\]])", r"\1", t[s:e + 1])
    return json.loads(s_str)

## 5. Run Evaluation & Export Report

In [ ]:
def render_placeholders(t):
    return str(t or "").replace("{{char}}", CHAR_NAME).replace("{{user}}", USER_NAME)

rng = random.Random(SEED)
sampled_pairs = list(test_pairs)
rng.shuffle(sampled_pairs)
sampled_pairs = sampled_pairs[:NUM_SAMPLES]

rendered_system = render_placeholders(SYSTEM_PROMPT)
rendered_refs = "\n".join(f"- {render_placeholders(r)}" for r in reference_samples)

items = []
for idx, pair in enumerate(sampled_pairs, start=1):
    ctx = render_placeholders(pair["konteks"])
    gt = render_placeholders(pair["balasan"])

    candidate = call_llm(
        messages=[{"role": "system", "content": rendered_system}, {"role": "user", "content": ctx}],
        model=CANDIDATE_MODEL,
        temperature=0.7,
        max_tokens=400,
    ).strip()

    judge_prompt = (
        JUDGE_USER_TEMPLATE
        .replace("{conversation_context}", ctx)
        .replace("{target_reference_samples}", rendered_refs)
        .replace("{ground_truth_reply}", gt)
        .replace("{candidate_response}", candidate)
    )
    raw_judge = call_llm(
        messages=[{"role": "system", "content": JUDGE_SYSTEM_PROMPT}, {"role": "user", "content": judge_prompt}],
        model=JUDGE_MODEL,
        temperature=0.1,
        max_tokens=1000,
    )
    scores = parse_json_loose(raw_judge)
    for k in SCORE_KEYS:
        scores[k] = max(1, min(5, int(round(float(scores.get(k, 3))))))
    scores["overall_score"] = round(sum(scores[k] for k in SCORE_KEYS) / 4.0, 2)

    items.append({
        "turn_index": idx,
        "conversation_context": ctx,
        "ground_truth_reply": gt,
        "candidate_response": candidate,
        "scores": scores,
    })
    print(f"[{idx}/{len(sampled_pairs)}] Overall: {scores['overall_score']}/5 | Style={scores['style_score']} Reg={scores['register_score']} Persona={scores['persona_score']} Coh={scores['coherence_score']}")

summary = {k: round(statistics.mean(it["scores"][k] for it in items), 2) for k in SCORE_KEYS}
summary["overall_score"] = round(sum(summary[k] for k in SCORE_KEYS) / 4.0, 2)
summary["fidelity_pct"] = round(summary["overall_score"] * 20.0, 1)

report = {"n_evaluated": len(items), "summary": summary, "items": items}
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))
pd.DataFrame([{
    "turn": it["turn_index"],
    "context": it["conversation_context"][:60],
    "ground_truth": it["ground_truth_reply"][:60],
    "candidate": it["candidate_response"][:60],
    **it["scores"],
} for it in items])